# Module 3 • Classical Natural Language Processing

# Lesson 15 • Bag-of-Words and TF-IDF Representation

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Beginner  
**Estimated study time:** 90–120 minutes

---

## Scope

This lesson explains how classical NLP systems convert documents into numeric
vectors using Bag-of-Words, term frequency, document frequency, inverse
document frequency, TF-IDF weighting, vector normalization, and cosine
similarity.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain the Bag-of-Words assumption;
- construct a vocabulary and document-term matrix;
- distinguish binary, count, and frequency-based representations;
- calculate term frequency and document frequency;
- explain inverse document frequency;
- calculate TF-IDF manually;
- interpret sparse feature matrices;
- compare raw counts and TF-IDF;
- calculate cosine similarity between documents;
- inspect learned features;
- use CountVectorizer and TfidfVectorizer safely;
- evaluate representation choices inside leakage-safe pipelines.

## Table of Contents

1. From Text to Numeric Vectors
2. The Bag-of-Words Assumption
3. Vocabulary Construction
4. Document-Term Matrices
5. Binary Features
6. Count Features
7. Term Frequency
8. Document Frequency
9. Inverse Document Frequency
10. TF-IDF
11. Manual TF-IDF Calculation
12. Scikit-Learn CountVectorizer
13. Scikit-Learn TfidfVectorizer
14. Vector Normalization
15. Cosine Similarity
16. Feature Interpretation
17. Vocabulary Pruning
18. Classification Pipelines
19. Limitations and Multilingual Considerations
20. Evaluation and Error Analysis
21. Knowledge Check
22. Exercises
23. Summary and Next Lesson

# 1. From Text to Numeric Vectors

Machine Learning algorithms operate on numeric features rather than raw text.

A representation function converts:

```text
"the model analyzes text"
```

into a vector such as:

```text
[1, 1, 1, 1, 0, 0, ...]
```

Each vector position corresponds to a vocabulary feature.

In [ ]:
documents = [
    "the model analyzes text",
    "the model analyzes documents",
    "the system processes text",
]

for index, document in enumerate(documents):
    print(index, document)

# 2. The Bag-of-Words Assumption

**Bag-of-Words (BoW)** represents a document using token occurrence while
ignoring most word order.

Compare:

```text
dog bites man
man bites dog
```

Both contain the same unigrams, so a unigram Bag-of-Words representation is
identical even though the meanings differ.

In [ ]:
from collections import Counter

sentence_a = "dog bites man"
sentence_b = "man bites dog"

print(Counter(sentence_a.split()))
print(Counter(sentence_b.split()))
print("Same unigram counts:", Counter(sentence_a.split()) == Counter(sentence_b.split()))

Bag-of-Words is simple and effective for many classical tasks, but it discards:

- long-range word order;
- syntax;
- discourse;
- compositional meaning;
- most contextual information.

# 3. Vocabulary Construction

A **vocabulary** maps each feature to a fixed vector position.

Example corpus:

```text
the model analyzes text
the system processes text
```

Possible sorted vocabulary:

```text
analyzes, model, processes, system, text, the
```

In [ ]:
tokenized_documents = [
    document.lower().split()
    for document in documents
]

vocabulary = sorted({
    token
    for document in tokenized_documents
    for token in document
})

vocabulary_index = {
    token: index
    for index, token in enumerate(vocabulary)
}

print("Vocabulary:", vocabulary)
print("Mapping:", vocabulary_index)

Vocabulary construction must be fitted on training data only. Adding test
tokens before evaluation leaks information about the test distribution.

# 4. Document-Term Matrices

A **document-term matrix** contains:

- one row per document;
- one column per vocabulary feature;
- one value per document-feature pair.

In [ ]:
import numpy as np
import pandas as pd

count_matrix_manual = np.zeros(
    (len(tokenized_documents), len(vocabulary)),
    dtype=int,
)

for document_index, tokens in enumerate(tokenized_documents):
    token_counts = Counter(tokens)

    for token, count in token_counts.items():
        count_matrix_manual[
            document_index,
            vocabulary_index[token],
        ] = count

pd.DataFrame(
    count_matrix_manual,
    columns=vocabulary,
    index=[f"doc_{i}" for i in range(len(documents))],
)

The matrix is often sparse because each document uses only a small fraction of
the complete vocabulary.

# 5. Binary Features

Binary features record only presence or absence.

```text
1 → feature occurs
0 → feature does not occur
```

Repeated occurrences do not increase the value.

In [ ]:
binary_matrix_manual = (
    count_matrix_manual > 0
).astype(int)

pd.DataFrame(
    binary_matrix_manual,
    columns=vocabulary,
    index=[f"doc_{i}" for i in range(len(documents))],
)

Binary features can be useful when occurrence matters more than frequency.

# 6. Count Features

Count features preserve how many times each token occurs.

Compare:

```text
good service
good good good service
```

Binary vectors treat `good` as present in both. Count vectors assign different
values.

In [ ]:
comparison_documents = [
    "good service",
    "good good good service",
]

comparison_counts = [
    Counter(document.split())
    for document in comparison_documents
]

for document, counts in zip(comparison_documents, comparison_counts):
    print(document, "->", counts)

Raw counts may overemphasize longer documents because they naturally contain
more token occurrences.

# 7. Term Frequency

**Term Frequency (TF)** measures a term's importance within one document.

A common normalized form is:

\[
TF(t,d) = rac{count(t,d)}{	ext{number of tokens in } d}
\]

where:

- \(t\) is a term;
- \(d\) is a document.

In [ ]:
def normalized_term_frequency(tokens: list[str]) -> dict[str, float]:
    counts = Counter(tokens)
    total = len(tokens)

    return {
        term: count / total
        for term, count in counts.items()
    }


for index, tokens in enumerate(tokenized_documents):
    print(f"Document {index}:", normalized_term_frequency(tokens))

Other TF definitions include raw count, logarithmic scaling, and binary
presence.

# 8. Document Frequency

**Document Frequency (DF)** counts how many documents contain a term.

\[
DF(t) = |\{d : t \in d\}|
\]

It is a document-level count, not a total occurrence count.

In [ ]:
document_frequency = {}

for term in vocabulary:
    document_frequency[term] = sum(
        term in set(tokens)
        for tokens in tokenized_documents
    )

pd.Series(
    document_frequency,
    name="Document frequency",
).sort_values(ascending=False)

Terms appearing in many documents often provide less discrimination between
documents.

# 9. Inverse Document Frequency

**Inverse Document Frequency (IDF)** gives higher weight to terms that occur
in fewer documents.

One common form is:

\[
IDF(t) = \log\left(rac{N}{DF(t)}ight)
\]

where \(N\) is the number of documents.

In [ ]:
import math

number_of_documents = len(tokenized_documents)

idf_manual = {
    term: math.log(
        number_of_documents / document_frequency[term]
    )
    for term in vocabulary
}

pd.Series(
    idf_manual,
    name="Unsmoothed IDF",
).sort_values(ascending=False)

Libraries often use smoothing to avoid zero divisions and to stabilize values.

# 10. TF-IDF

**TF-IDF** combines local importance and corpus rarity:

\[
TFIDF(t,d) = TF(t,d) 	imes IDF(t)
\]

A term receives a high value when it is:

- frequent in one document;
- uncommon across the corpus.

TF-IDF does not measure semantic meaning. It measures statistical importance
under a specific corpus and weighting scheme.

# 11. Manual TF-IDF Calculation

In [ ]:
tfidf_manual = np.zeros(
    (len(tokenized_documents), len(vocabulary)),
    dtype=float,
)

for document_index, tokens in enumerate(tokenized_documents):
    tf_values = normalized_term_frequency(tokens)

    for term, tf_value in tf_values.items():
        tfidf_manual[
            document_index,
            vocabulary_index[term],
        ] = tf_value * idf_manual[term]

manual_tfidf_frame = pd.DataFrame(
    tfidf_manual,
    columns=vocabulary,
    index=[f"doc_{i}" for i in range(len(documents))],
)

manual_tfidf_frame.round(3)

The term `the` receives a low or zero unsmoothed IDF because it appears in all
documents. Rarer terms receive larger weights.

# 12. Scikit-Learn CountVectorizer

`CountVectorizer` constructs a vocabulary and count matrix.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer()

count_matrix = count_vectorizer.fit_transform(documents)

print("Feature names:")
print(count_vectorizer.get_feature_names_out())
print("Matrix shape:", count_matrix.shape)

In [ ]:
count_frame = pd.DataFrame(
    count_matrix.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=[f"doc_{i}" for i in range(len(documents))],
)

count_frame

`CountVectorizer(binary=True)` creates binary occurrence features.

In [ ]:
binary_vectorizer = CountVectorizer(binary=True)
binary_matrix = binary_vectorizer.fit_transform(documents)

pd.DataFrame(
    binary_matrix.toarray(),
    columns=binary_vectorizer.get_feature_names_out(),
    index=[f"doc_{i}" for i in range(len(documents))],
)

# 13. Scikit-Learn TfidfVectorizer

`TfidfVectorizer` combines token counting, IDF weighting, and vector
normalization.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

print("Feature names:")
print(tfidf_vectorizer.get_feature_names_out())
print("Matrix shape:", tfidf_matrix.shape)

In [ ]:
tfidf_frame = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"doc_{i}" for i in range(len(documents))],
)

tfidf_frame.round(3)

Scikit-learn uses smoothed IDF and L2 normalization by default, so its values
differ from the earlier manual unsmoothed example.

In [ ]:
sklearn_idf = pd.Series(
    tfidf_vectorizer.idf_,
    index=tfidf_vectorizer.get_feature_names_out(),
    name="Scikit-learn IDF",
)

sklearn_idf.sort_values(ascending=False)

# 14. Vector Normalization

Vector normalization reduces the influence of document length.

Common norms include:

- L1: sum of absolute values equals 1;
- L2: Euclidean length equals 1;
- no normalization.

In [ ]:
l2_lengths = np.sqrt(
    tfidf_matrix.multiply(tfidf_matrix).sum(axis=1)
)

print("L2 lengths:")
print(np.asarray(l2_lengths).ravel().round(3))

L2 normalization is common before cosine similarity because normalized dot
products become directly comparable.

# 15. Cosine Similarity

Cosine similarity measures the angle between two vectors.

\[
cosine(a,b) =
rac{a \cdot b}{||a||\,||b||}
\]

Values close to 1 indicate similar directions.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

pd.DataFrame(
    similarity_matrix,
    index=[f"doc_{i}" for i in range(len(documents))],
    columns=[f"doc_{i}" for i in range(len(documents))],
).round(3)

Similarity reflects shared weighted features, not guaranteed semantic
equivalence.

## 15.1 Query-to-Document Similarity

In [ ]:
query = ["model analyzes text"]
query_vector = tfidf_vectorizer.transform(query)

query_scores = cosine_similarity(
    query_vector,
    tfidf_matrix,
).ravel()

ranking = pd.DataFrame(
    {
        "Document": documents,
        "Cosine similarity": query_scores,
    }
).sort_values(
    "Cosine similarity",
    ascending=False,
)

ranking

The query must be transformed with the already fitted vectorizer. Refitting on
the query would create a different feature space.

# 16. Feature Interpretation

Classical linear models can expose feature weights.

Positive and negative coefficients indicate how strongly features support each
class, subject to model and data limitations.

In [ ]:
training_texts = [
    "excellent service",
    "helpful support",
    "fast response",
    "friendly staff",
    "terrible service",
    "rude support",
    "slow response",
    "unhelpful staff",
]

training_labels = [
    "positive",
    "positive",
    "positive",
    "positive",
    "negative",
    "negative",
    "negative",
    "negative",
]

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

interpretation_pipeline = Pipeline(
    [
        ("tfidf", TfidfVectorizer()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

interpretation_pipeline.fit(
    training_texts,
    training_labels,
)

feature_names = (
    interpretation_pipeline
    .named_steps["tfidf"]
    .get_feature_names_out()
)

coefficients = (
    interpretation_pipeline
    .named_steps["classifier"]
    .coef_[0]
)

coefficient_frame = pd.DataFrame(
    {
        "Feature": feature_names,
        "Coefficient": coefficients,
    }
).sort_values("Coefficient")

coefficient_frame

Coefficients describe the fitted model, not universal word sentiment.

# 17. Vocabulary Pruning

Vocabulary size can be controlled using:

- `min_df`;
- `max_df`;
- `max_features`;
- stop-word policies;
- n-gram range;
- token pattern.

In [ ]:
pruning_documents = [
    "the model analyzes text",
    "the model analyzes documents",
    "the system processes text",
    "rareterm appears once",
]

full = CountVectorizer()
pruned = CountVectorizer(min_df=2)

full.fit(pruning_documents)
pruned.fit(pruning_documents)

print("Full vocabulary:")
print(full.get_feature_names_out())

print("\nPruned vocabulary:")
print(pruned.get_feature_names_out())

Aggressive pruning may remove rare but important domain terms.

# 18. Classification Pipelines

Count and TF-IDF representations should be compared inside the same
cross-validation workflow.

In [ ]:
classification_texts = [
    "excellent service and helpful staff",
    "helpful and fast response",
    "friendly customer support",
    "excellent assistance",
    "terrible service and rude staff",
    "slow and unhelpful response",
    "rude customer support",
    "terrible assistance",
    "fast helpful service",
    "slow rude service",
    "friendly support",
    "unhelpful staff",
]

classification_labels = [
    "positive",
    "positive",
    "positive",
    "positive",
    "negative",
    "negative",
    "negative",
    "negative",
    "positive",
    "negative",
    "positive",
    "negative",
]

In [ ]:
from sklearn.model_selection import cross_val_score

count_pipeline = Pipeline(
    [
        ("vectorizer", CountVectorizer()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

tfidf_pipeline = Pipeline(
    [
        ("vectorizer", TfidfVectorizer()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

count_scores = cross_val_score(
    count_pipeline,
    classification_texts,
    classification_labels,
    cv=3,
    scoring="f1_macro",
)

tfidf_scores = cross_val_score(
    tfidf_pipeline,
    classification_texts,
    classification_labels,
    cv=3,
    scoring="f1_macro",
)

pd.DataFrame(
    {
        "Representation": ["Counts", "TF-IDF"],
        "Mean macro F1": [
            count_scores.mean(),
            tfidf_scores.mean(),
        ],
    }
)

The corpus is intentionally small. The result demonstrates method comparison,
not a general claim that one representation is always superior.

# 19. Limitations and Multilingual Considerations

Bag-of-Words and TF-IDF have important limitations:

- weak representation of word order;
- no inherent semantics;
- fixed vocabulary;
- sparse high-dimensional vectors;
- sensitivity to tokenization;
- poor handling of unseen features;
- corpus-dependent IDF values.

## 19.1 Arabic Considerations

Arabic vectorization depends on:

- diacritics policy;
- spelling normalization;
- clitic segmentation;
- stemming or lemmatization;
- Modern Standard Arabic versus dialect;
- code-switching;
- word versus character n-grams.

Character TF-IDF may reduce sensitivity to segmentation and spelling variation,
but it does not replace morphological analysis.

In [ ]:
arabic_documents = [
    "الكتاب مفيد",
    "والكتاب مفيد",
    "كتابه مفيد",
    "الدرس مفيد",
]

arabic_word_tfidf = TfidfVectorizer(
    analyzer="word",
).fit_transform(arabic_documents)

arabic_char_tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 4),
).fit_transform(arabic_documents)

print("Arabic word TF-IDF shape:", arabic_word_tfidf.shape)
print("Arabic character TF-IDF shape:", arabic_char_tfidf.shape)

Different feature shapes reflect different representations, not automatically
better quality.

# 20. Evaluation and Error Analysis

Representation choices may be evaluated through:

- macro and weighted F1;
- precision and recall;
- retrieval ranking metrics;
- cosine similarity quality;
- vocabulary size;
- matrix density;
- memory use;
- training time;
- performance by document length, domain, and language.

## Common Errors

- vocabulary fitted before train-test splitting;
- query vectorizer refitted separately;
- tokens removed without task justification;
- document length dominating raw counts;
- rare domain terms pruned;
- test identifiers entering the vocabulary;
- inconsistent preprocessing;
- interpreting TF-IDF as semantic understanding;
- comparing matrices built from different vocabularies;
- ignoring Arabic normalization or segmentation.

In [ ]:
representation_errors = pd.DataFrame(
    [
        ("Fit vocabulary on all data", "test distribution leakage"),
        ("Refit vectorizer on query", "incompatible feature space"),
        ("Use raw counts for long documents", "length bias"),
        ("Prune rare medical terms", "domain information loss"),
        ("Interpret high cosine as equivalence", "semantic overclaim"),
    ],
    columns=["Decision", "Risk"],
)

representation_errors

# 21. Knowledge Check

1. What does Bag-of-Words represent?
2. Which information does unigram BoW discard?
3. What is a vocabulary?
4. What is a document-term matrix?
5. How do binary and count features differ?
6. What is term frequency?
7. What is document frequency?
8. Why does IDF down-weight common terms?
9. What does TF-IDF combine?
10. Why is vector normalization useful?
11. What does cosine similarity measure?
12. Why must a query use the fitted vectorizer?
13. How can vocabulary pruning help?
14. Why must vectorization remain inside a pipeline?
15. Which Arabic preprocessing decisions affect TF-IDF?

# 22. Exercises

## Exercise 1 — Manual BoW

Construct a vocabulary and count matrix for five sentences without using a
library.

## Exercise 2 — Binary Versus Count

Compare binary and count features for documents containing repeated words.

## Exercise 3 — Manual TF

Calculate normalized term frequency for each term in three documents.

## Exercise 4 — Manual IDF

Calculate unsmoothed and smoothed IDF for a small corpus.

## Exercise 5 — TF-IDF

Reproduce a TF-IDF matrix manually and compare it with scikit-learn.

## Exercise 6 — Similarity

Rank documents for five queries using cosine similarity.

## Exercise 7 — Classification

Compare count and TF-IDF Logistic Regression pipelines using repeated
cross-validation.

## Exercise 8 — Arabic Representation

Compare Arabic word TF-IDF, character TF-IDF, and normalized word TF-IDF.

## Challenge Exercises

1. Implement a complete TF-IDF transformer from scratch.
2. Compare L1, L2, and no normalization.
3. Add word bigrams and compare retrieval quality.
4. Build a feature-inspection report for a trained classifier.
5. Measure vocabulary size, density, training time, and macro F1 across several
   vectorizer configurations.

# 23. Summary and Next Lesson

In this lesson:

- Bag-of-Words converted documents into vocabulary-based vectors;
- binary features represented presence;
- count features represented frequency;
- term frequency measured local importance;
- document frequency measured corpus coverage;
- inverse document frequency down-weighted common terms;
- TF-IDF combined local frequency with corpus rarity;
- sparse matrices stored high-dimensional text efficiently;
- vector normalization reduced document-length effects;
- cosine similarity compared vector directions;
- fitted vocabularies and feature weights remained inside pipelines;
- feature inspection supported model interpretation;
- vocabulary pruning controlled dimensionality;
- Arabic representation depended on normalization, segmentation, and n-gram
  policy.

## Next Lesson

**Lesson 16: Classical Text Classification with Naive Bayes, Logistic
Regression, and Support Vector Machines** compares major classifiers using
leakage-safe text pipelines and appropriate evaluation metrics.

# References

- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.
- Manning, C. D., Raghavan, P., & Schütze, H. *Introduction to Information Retrieval*.
- scikit-learn text feature extraction documentation.
- classical information-retrieval and text-classification literature.